# Exercise 08

## Changes to the Evacuation Model

To enable exercises about sensing and interaction, the evacuation model has been modified:

* New model and agent parameter `maxsight`
* Changes in `agent.explore_fieldofvision` to consider `maxsight`
* New switch `DISTANCE_NOISE` to add noise to the calculation of distances to exits (`* self.model.rng.normal(loc=1.0, scale=2.0)`)
* New model parameters `interact_neumann2`, `interact_moore2` and `interact_swnetwork` and agent parameter `interactionmatrix` (combining the three before-mentioned), which indicate the probability of passing alarm message via the particular topology.
* When one of `interact_neumann2`, `interact_moore2` and `interact_swnetwork` is not `None` initially only one agent knows about the alarm.
* With a probabilty of 0.01 each agent who is not alarmed gets alarmed (otherwise the room would never be evacuated if an agent is not informed).
* Adding `Network`

## Evaluation Code


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import itertools
import time
from datetime import timedelta
from matplotlib.colors import LinearSegmentedColormap

import sys
sys.path.insert(0,'../../abmodel')

from fire_evacuation.model import FireEvacuation, FireEvacuationScenario
from fire_evacuation.agent import Human

unikcolors = [np.array((80,149,200))/255, np.array((74,172,150))/255,
                                                  np.array((234,195,114))/255, np.array((199,16,92))/255]
uniks = LinearSegmentedColormap.from_list( 'unik', unikcolors)

%matplotlib inline

In [ ]:
def batchrun_model(
        scenario,
        experiments,
        replications=20,
        aggregate_output_per_run=True,
    ):
    """
    Instantiates Scenarios for parameter/replication variations, runs the model
    for every Scenario and concats results in a pd.DataFrame.

    Parameters
    ----------
    experiments: pd.DataFrame or dict
        data frame of parameter combinations with parameter names as column names
    replications: int
        number of runs with different RNG
    max_step: int
        the number of steps the model is run in case it does not terminate earlier
    aggregate_output_per_run: bool
        calculate mean values (except for Step)
    """
    # unfortunately, scenario.from_dataframe() is a class method and does not consider values in scenario,
    # so we need to add them to experiments manually:
    experiments = {key: [value] for key, value in scenario.__dict__.items()} | experiments
    
    # create combinations of given parameter values
    if type(experiments) is pd.DataFrame:
        experiments = pd.DataFrame(itertools.product(*[experiments[col] for col in experiments.columns]),
                               columns=experiments.columns)
    elif type(experiments) is dict:
        experiments = pd.DataFrame(itertools.product(*experiments.values()),
                               columns=experiments.keys())
    
    scenarios = scenario.from_dataframe(
            experiments=experiments,
            replications=replications,
    )
    results = pd.DataFrame()
    
    print(f"Perform {len(scenarios)} model runs...")
    start = time.time()
    for scenario in scenarios:
        model =  FireEvacuation(scenario=scenario)
        model.run_model()
        d = model.datacollector.get_model_vars_dataframe()
        d.insert(0, "sid", scenario.scenario_id)
        d.insert(0, "rid", scenario.replication_id)
        d.index.name="Step"
        d = d.reset_index()
        # calc means
        if aggregate_output_per_run:
            d = d.groupby(["sid", "rid"]).agg(
                {column:"mean" if column != "Step" else "max" for column in d.columns if column not in ["sid", "rid"]}
            )
        d = pd.concat([pd.DataFrame(scenario.__dict__, index=d.index), d], axis=1)
        results = pd.concat([results, d])
    print(f"Done! Took {str(timedelta(seconds=time.time() - start))} h")
    return results

# Task 2 (Sensing in the evacuation model)

## Subtask 2.2

We can change an agents sensing by limiting the field of vision, which is applied when agents search for exits and others to help. Explore the model's sensitivity towards the `maxsight` parameter with batch runs. Produce a figure (consult e.g. exercise 07 task 2.4) showing final values of `TurnCount` and `Step` for increasing values of `maxsight`. Discuss (i.e. contextualise, compare and explain your observations) the results. (<200 words)

In [ ]:
import numpy as np

results = batchrun_model(
    scenario = FireEvacuationScenario(
        floor_size=14,
        human_count=100,
        nervousness_mean = 0.3,
        alarm_believers_prop=0.9,
        rng=7,
    ),
    experiments = {
        "maxsight": {2,4,6,8,10,12,14,16},
    },
    replications= 20,
    aggregate_output_per_run=True,
)

In [ ]:
data1 = pd.DataFrame(results.reset_index())[['maxsight', "TurnCount", 'Step', 'rid']].round(decimals=1)
data1.groupby(['rid','maxsight']).agg("max").groupby(['maxsight']).agg("mean")

In [ ]:
# Some hints to produce a comparison figure

# remove columns not needed
d_sw.drop(["interact_moore", "interact_neumann"], axis=1, inplace=True)

# rename column to align it to other frames
d_sw.rename(columns={"interact_swnetwork": "interact"}, inplace=True)

# merge DataFrames on 'column' using union of keys from both frames 
plotdata = pd.merge(df1, df2, how='outer', on='column', suffixes=("_df1", "_df2"))

# define a column as index (for x-axis of plot)
plotdata.set_index('column', inplace=True)

## Subtask 2.3

Sensing is often not 100% accurate - uncertainty is involved. A way to represent uncertainty in agent-based modelling is to introduce noise - a random factor that reduces accuracy of the perceived value. Implement random noise for sensing the distance between the agent and a certain exit. Use the (existing) agent parameter switch to enable/disable noise and define a parameter to calibrate the level of noise. Consider agent heterogeneity! Test your implementation. Does it change results significantly (t-test)?

In [ ]:
results = batchrun_model(
    scenario = FireEvacuationScenario(
        floor_size=14,
        human_count=70,
        nervousness_mean = 0.3,
        alarm_believers_prop=0.5,
        distancenoisefactor = 0.4,
        rng=7,
    ),
    experiments = {
        "distancenoise": {False,True},
    },
    replications= 20,
    aggregate_output_per_run=False,
)

data = pd.DataFrame(results)[['distancenoise', 'Step', 'rid']].round(decimals=1)
db = data.groupby(['rid','distancenoise']).agg("max").groupby(['distancenoise']).agg("mean").reset_index()

## Subtask 2.4

Find two more processes of agents' sensing that could be modelled with uncertainty. How would you introduce uncertainty (provide pseudo code)? Discuss whether you should add this option to the model code (<200 words)!

# Task 3 (Interaction in the evacuation model)

## Subtask 3.1

A new option was added to the model: Assume the fire alarm is broken or not existing, and a single randomly choosen agent gets aware of an incident in the room that requires all to escape. We investigate different ways of interaction in terms of passing the information of fire alarm:

* Propagation in the von-Neumann-neighbourhood 
* Propagation in the Moore-Neighbourhood 
* Propagation on a Small-World-Network (Watts-Beta) 

The model parameters determine the probability by which the information is 	passed from an informed agent to its neighbours on the particular topology.
Execute the code blocks in the notebook and compare the results in a figure (!) and interprete them! For small-world networks, why is the <ins>difference</ins> between low values of 	propagation probability higher than for higher values of the probability?

In [ ]:
# Von Neumann:   
results_vn = batchrun_model(
    scenario = FireEvacuationScenario(
        floor_size=14,
        human_count=100,
        nervousness_mean = 0.3,
        cooperation_mean = 0.1,
        distancenoisefactor = 0.4,
        rng=7,
    ),
    experiments = {
        "interact_neumann": {0.01,0.02,0.05,0.1,0.5,1.0},
        "interact_moore": {0.0},
        "interact_swnetwork": {0.0},
    },
    replications= 20,
    aggregate_output_per_run=False,
)

data_vn = pd.DataFrame(results_vn)[['interact_neumann', 'interact_moore','interact_swnetwork',
                              'Step', 'TurnCount', 'rid']].round(decimals=2)
db_vn = data_vn.groupby(['rid','interact_neumann','interact_moore','interact_swnetwork',]).agg("max").\
        groupby(['interact_neumann','interact_moore','interact_swnetwork']).agg("mean")
db_vn

In [ ]:
# Moore:   
results_mr = batchrun_model(
    scenario = FireEvacuationScenario(
        floor_size=14,
        human_count=100,
        nervousness_mean = 0.3,
        cooperation_mean = 0.1,
        distancenoisefactor = 0.4,
        rng=7,
    ),
    experiments = {
        "interact_neumann": {0.0},
        "interact_moore": {0.01,0.02,0.05,0.1,0.5,1.0},
        "interact_swnetwork": {0.0},
    },
    replications= 20,
    aggregate_output_per_run=False,
)

data_mr = pd.DataFrame(results_mr)[['interact_neumann', 'interact_moore','interact_swnetwork',
                              'Step', 'TurnCount', 'rid']].round(decimals=2)
db_mr = data_mr.groupby(['rid','interact_neumann','interact_moore','interact_swnetwork',]).agg("max").\
        groupby(['interact_neumann','interact_moore','interact_swnetwork']).agg("mean")
db_mr

In [ ]:
# Small-world network:   
results_sw = batchrun_model(
    scenario = FireEvacuationScenario(
        floor_size=14,
        human_count=100,
        nervousness_mean = 0.3,
        cooperation_mean = 0.1,
        alarm_believers_prop=0.5,
        distancenoisefactor = 0.4,
        rng=7,
    ),
    experiments = {
        "interact_neumann": {0.0},
        "interact_moore": {0.0},
        "interact_swnetwork": {0.01,0.02,0.05,0.1,0.5,1.0},
    },
    replications= 20,
    aggregate_output_per_run=False,
)

data_sw = pd.DataFrame(results_sw)[['interact_neumann', 'interact_moore','interact_swnetwork',
                              'Step', 'TurnCount', 'rid']].round(decimals=2)
db_sw = data_sw.groupby(['rid','interact_neumann','interact_moore','interact_swnetwork',]).agg("max").\
        groupby(['interact_neumann','interact_moore','interact_swnetwork']).agg("mean")
db_sw

In [ ]:
# Some hints to produce a figure

# remove columns not needed
d_sw.drop(["interact_moore", "interact_neumann"], axis=1, inplace=True)

# rename column to align it to other frames
d_sw.rename(columns={"interact_swnetwork": "interact"}, inplace=True)

# merge DataFrames on 'column' using union of keys from both frames 
plotdata = pd.merge(df1, df2, how='outer', on='column', suffixes=("_df1", "_df2"))

# define a column as index (for x-axis of plot)
plotdata.set_index('column', inplace=True)

## Subtask 3.2

Which initial position would be ideal for each single interaction topology (vonNeumann, Moore, Network)? Format your answer either as list or table.

## Subtask 3.3

Implement placing the initial knowing agent at a beneficial position for propagation on the smallworld-network. Consider [this answer](https://stackoverflow.com/a/58392749/3957413)!

First describe your idea here in pseudo code, then implement in model.py at line 356ff. Copy and re-run the last part of 3.1 (values for `interact_swnetwork`).